# Deliverable 4 — Model Development & Evaluation

**Objective:** Compare a seasonal-naive baseline, Random Forest, and XGBoost to predict route-level monthly `OD_PAX` as far as **24 months ahead**. All features must be available at the forecast origin, and evaluation uses a chronological holdout.

**Privacy:** Run locally on authorized data only. Outputs and generated CSVs can contain confidential AA information; clear outputs before sharing/committing. This notebook does **not** upload or commit any data.

**Dependency:** First run `deliverable3/notebooks/02-feature-engineering.ipynb` to produce `data/processed/aa_features_24m.csv`.
**Experiment note:** This version enables the new market-aware features. The prior test period has been inspected; report improvements as exploratory unless confirmed on a separate untouched holdout.


## 1 — Imports and configuration
Select your project `.venv` kernel. Install missing packages into that environment using your terminal if needed: `python -m pip install pandas numpy scikit-learn matplotlib xgboost`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.base import clone

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost not installed; install xgboost to include that comparison.")

root = Path.cwd().resolve()
while not (root / ".git").exists():
    if root == root.parent:
        raise FileNotFoundError("Launch notebook inside the git repository.")
    root = root.parent

FEATURE_PATH = root / "data/processed/aa_features_24m.csv"
OUTPUT_DIR = root / "data/processed"
HOLDOUT_MONTHS = 24  # Target a single 24-month future window
VALIDATION_MONTHS = 12
RANDOM_STATE = 42
print("Repository:", root)
print("Processed feature file exists:", FEATURE_PATH.exists())

## 2 — Load data, validate required columns, and build a strict allowlist

In [ ]:
if not FEATURE_PATH.exists():
    raise FileNotFoundError(f"Run 02-feature-engineering.ipynb first: {FEATURE_PATH}")

df = pd.read_csv(FEATURE_PATH)
required = {"ROUTE", "DEP_YEAR", "DEP_MONTH", "OD_PAX", "LAG_24"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing expected processed columns: {sorted(missing)}")

if "DATE" in df.columns:
    df["DATE"] = pd.to_datetime(df["DATE"], errors="raise")
else:
    df["DATE"] = pd.to_datetime(dict(year=df.DEP_YEAR, month=df.DEP_MONTH, day=1))
df["DATE"] = df["DATE"].dt.to_period("M").dt.to_timestamp()
df["ROUTE"] = df["ROUTE"].astype(str)
df["OD_PAX"] = pd.to_numeric(df["OD_PAX"], errors="raise")
if df.duplicated(["ROUTE", "DATE"]).any():
    raise ValueError("Duplicate ROUTE–DATE keys: resolve in feature engineering.")
if df["OD_PAX"].isna().any() or (df["OD_PAX"] < 0).any():
    raise ValueError("OD_PAX has missing or negative values.")

# Target-relative lags >=24 are available by the start of a 24-month window
candidate_features = [
    "ROUTE", "DEP_YEAR", "DEP_MONTH", "MONTH_SIN", "MONTH_COS",
    "LAG_24", "LAG_25", "LAG_26", "LAG_36", "LAG_48",
    "RECENT_HIST_MEAN_3", "HIST_CHANGE_12M", "HIST_GROWTH_12M", "HAS_LAG_24",
    "ROUTE_VOLATILITY_24M", "ROUTE_MOMENTUM_12M",
    "ROUTE_SEASONALITY_INDEX", "ROUTE_HISTORY_COVERAGE"
]
features = [x for x in candidate_features if x in df.columns]
if "ROUTE" not in features or "LAG_24" not in features:
    raise ValueError("Feature set missing key columns.")
for c in features:
    if c != "ROUTE":
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Exclude same-month revenue, timestamps, row IDs, and anything not on allowlist.
df = df.sort_values(["DATE", "ROUTE"]).reset_index(drop=True)
print("Rows:", len(df), "Routes:", df.ROUTE.nunique())
print("Feature columns:", features)
print("Dates:", df.DATE.min().date(), "to", df.DATE.max().date())
print("Missing LAG_24:", int(df.LAG_24.isna().sum()))

## 3 — Feature availability and alignment checks

**Fixed 24-month horizon:** For a forecast issued at the end of month `C`, future target months are `C+1` through `C+24`. For any such target, `LAG_24` and older lags are available by `C` (provided observations exist). No 3- or 12-month lags are allowed here.

The precomputed `LAG_24` is verified against source records. Other engineered historical features should be audited in the feature-engineering notebook as well.

In [ ]:
lookup = df.set_index(["ROUTE", "DATE"])["OD_PAX"]
expected_lag24 = pd.Series(
    [lookup.get((route, date - pd.DateOffset(months=24)), np.nan)
     for route, date in zip(df["ROUTE"], df["DATE"])], index=df.index, dtype=float
)
actual_lag24 = df["LAG_24"].astype(float)
valid = np.isclose(actual_lag24, expected_lag24, equal_nan=True, rtol=1e-8, atol=1e-8)
if not valid.all():
    raise AssertionError(f"Calendar lag alignment failed for {(~valid).sum()} records.")
print("PASS — LAG_24 matches exact route-calendar history.")
print("PASS — No same-month revenue or other non-allowlisted columns used.")

## 4 — Chronological holdout and validation

Reserve the **last 24 calendar months** as a simulated forecast window. The training dataset contains only earlier targets. Within that training history, the latest 12 calendar months form a validation window for model comparison. **Do not tune based on holdout results.**

A dataset with insufficient history raises a clear error rather than silently changing the forecasting objective.

In [ ]:
all_months = pd.date_range(df.DATE.min(), df.DATE.max(), freq="MS")
if len(all_months) < HOLDOUT_MONTHS + VALIDATION_MONTHS + 12:
    raise ValueError("Insufficient time span for 24-month holdout, 12-month validation, and meaningful training. Review the date coverage.")
holdout_start = all_months[-HOLDOUT_MONTHS]
val_start = holdout_start - pd.DateOffset(months=VALIDATION_MONTHS)
train = df[df.DATE < val_start].copy()
val = df[(df.DATE >= val_start) & (df.DATE < holdout_start)].copy()
train_val = df[df.DATE < holdout_start].copy()
test = df[df.DATE >= holdout_start].copy()
for name, part in [("Train", train), ("Validation", val), ("Train+validation", train_val), ("Holdout", test)]:
    print(name, len(part), "rows", (str(part.DATE.min().date()) if len(part) else "none"), "to", (str(part.DATE.max().date()) if len(part) else "none"))
if min(len(train), len(val), len(test)) == 0:
    raise ValueError("One of the chronological sets is empty.")
print("Forecast issuance simulated at end of:", (holdout_start - pd.DateOffset(months=1)).date())

## 5 — Baseline and evaluation functions
The baseline predicts last observed same-calendar-month demand from 24 months ago. It is only evaluated where `LAG_24` exists; model metrics are also compared **on the same eligible rows** for fairness.

In [ ]:
def score(y, prediction):
    y = np.asarray(y, dtype=float)
    p = np.clip(np.asarray(prediction, dtype=float), 0, None)
    abs_error = np.abs(y - p)
    total = np.sum(np.abs(y))
    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": float(np.sqrt(mean_squared_error(y, p))),
        "WAPE_pct": float(100 * abs_error.sum() / total) if total > 0 else np.nan,
        "n": len(y),
    }

def eligible_subset(part):
    # Comparable baseline-and-model rows. Never replace missing history with target values.
    return part.loc[part["LAG_24"].notna()].copy()

numeric = [c for c in features if c != "ROUTE"]
transform = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=False), numeric),
    ("route", OneHotEncoder(handle_unknown="ignore"), ["ROUTE"]),
])
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=1,
        max_features=0.7,
        random_state=42,
        n_jobs=-1),
}
if HAS_XGBOOST:
    models["XGBoost"] = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.07,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.9,
        reg_lambda=5,
        random_state=42,
        n_jobs=-1
    )
print("Models:", list(models))

## 6 — Validation comparison (choose model before looking at holdout)

In [ ]:
val_eligible = eligible_subset(val)
if len(val_eligible) == 0:
    raise ValueError("No validation rows have LAG_24. Check coverage; do not fabricate missing lags.")
validation_scores = []
val_preds = {"Seasonal naive (24m)": val_eligible["LAG_24"].to_numpy()}
validation_scores.append({"model": "Seasonal naive (24m)", **score(val_eligible.OD_PAX, val_preds["Seasonal naive (24m)"])})
for name, estimator in models.items():
    pipeline = Pipeline([("prep", clone(transform)), ("model", clone(estimator))])
    pipeline.fit(train[features], train.OD_PAX)
    pred = np.clip(pipeline.predict(val_eligible[features]), 0, None)
    val_preds[name] = pred
    validation_scores.append({"model": name, **score(val_eligible.OD_PAX, pred)})
validation_table = pd.DataFrame(validation_scores).sort_values("MAE")
print(validation_table.to_string(index=False))
winner = str(validation_table.iloc[0]["model"])
print("Selected using validation MAE only:", winner)

In [ ]:
from itertools import product

tuning_results = []

# Test 12 combinations
for depth, lr, trees in product(
    [2, 3, 4],
    [0.03, 0.07],
    [150, 300]
):
    candidate = XGBRegressor(
        n_estimators=trees,
        max_depth=depth,
        learning_rate=lr,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.9,
        reg_lambda=5,
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ("prep", clone(transform)),
        ("model", candidate)
    ])

    pipeline.fit(train[features], train["OD_PAX"])

    preds = np.clip(
        pipeline.predict(val_eligible[features]),
        0, None
    )

    metrics = score(val_eligible["OD_PAX"], preds)

    tuning_results.append({
        "depth": depth,
        "learning_rate": lr,
        "trees": trees,
        **metrics
    })

tuning_table = pd.DataFrame(tuning_results)
tuning_table = tuning_table.sort_values("MAE")

print("TOP 5 CONFIGURATIONS")
print(tuning_table.head(5).round(2).to_string(index=False))

print("\nPrevious XGBoost validation MAE: 177.27")
print("Best tuned MAE:", round(tuning_table.iloc[0]["MAE"], 2))

In [ ]:
# XGBoost feature ablation study — VALIDATION ONLY
from sklearn.base import clone

market_features = [
    "ROUTE_VOLATILITY_24M",
    "ROUTE_MOMENTUM_12M",
    "ROUTE_SEASONALITY_INDEX",
    "ROUTE_HISTORY_COVERAGE"
]

experiments = {
    "Original features": [
        f for f in features if f not in market_features
    ],
    "All features": features.copy()
}

# Remove one feature at a time
for feature in market_features:
    if feature in features:
        experiments[f"Without {feature}"] = [
            f for f in features if f != feature
        ]

ablation_results = []

for name, cols in experiments.items():

    numeric_cols = [c for c in cols if c != "ROUTE"]

    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        ("route", OneHotEncoder(handle_unknown="ignore"), ["ROUTE"])
    ])

    pipeline = Pipeline([
        ("prep", prep),
        ("model", clone(models["XGBoost"]))
    ])

    pipeline.fit(train[cols], train["OD_PAX"])

    preds = np.clip(
        pipeline.predict(val_eligible[cols]), 0, None
    )

    metrics = score(val_eligible["OD_PAX"], preds)

    ablation_results.append({
        "Experiment": name,
        **metrics
    })

ablation_table = pd.DataFrame(ablation_results)
ablation_table = ablation_table.sort_values("MAE")

print("FEATURE ABLATION RESULTS")
print(ablation_table.round(2).to_string(index=False))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from itertools import product

rf_results = []

for depth, leaf, max_features in product(
    [5, 10, None],
    [1, 3, 5],
    [0.7, 1.0]
):
    candidate = RandomForestRegressor(
        n_estimators=300,
        max_depth=depth,
        min_samples_leaf=leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ("prep", clone(transform)),
        ("model", candidate)
    ])

    pipeline.fit(train[features], train["OD_PAX"])

    preds = np.clip(
        pipeline.predict(val_eligible[features]),
        0, None
    )

    metrics = score(val_eligible["OD_PAX"], preds)

    rf_results.append({
        "max_depth": depth,
        "min_samples_leaf": leaf,
        "max_features": max_features,
        **metrics
    })

rf_table = pd.DataFrame(rf_results).sort_values("MAE")

print("TOP 5 RANDOM FOREST CONFIGURATIONS")
print(rf_table.head(5).round(2).to_string(index=False))

In [ ]:
# Rolling Time-Series Validation
# Uses historical periods only — excludes final holdout

rolling_results = []

# Two historical 12-month validation windows
for fold in [2, 1]:

    val_end = holdout_start - pd.DateOffset(months=12 * (fold - 1))
    val_begin = val_end - pd.DateOffset(months=12)

    roll_train = df[df["DATE"] < val_begin].copy()

    roll_val = df[
        (df["DATE"] >= val_begin) &
        (df["DATE"] < val_end)
    ].copy()

    roll_val = eligible_subset(roll_val)

    print(f"\nFold {3-fold}: {val_begin.date()} to "
          f"{(val_end - pd.DateOffset(months=1)).date()}")

    for name, estimator in models.items():

        pipeline = Pipeline([
            ("prep", clone(transform)),
            ("model", clone(estimator))
        ])

        pipeline.fit(
            roll_train[features],
            roll_train["OD_PAX"]
        )

        preds = np.clip(
            pipeline.predict(roll_val[features]),
            0, None
        )

        metrics = score(roll_val["OD_PAX"], preds)

        rolling_results.append({
            "Fold": 3 - fold,
            "Model": name,
            **metrics
        })

    # Seasonal baseline
    metrics = score(
        roll_val["OD_PAX"],
        roll_val["LAG_24"]
    )

    rolling_results.append({
        "Fold": 3 - fold,
        "Model": "Seasonal naive (24m)",
        **metrics
    })

rolling_table = pd.DataFrame(rolling_results)

print("\nROLLING VALIDATION RESULTS")
print(rolling_table.round(2).to_string(index=False))

print("\nAVERAGE VALIDATION MAE")
print(
    rolling_table.groupby("Model")["MAE"]
    .mean()
    .sort_values()
    .round(2)
)

## 7 — Refit the selected model and evaluate once on the untouched holdout

In [ ]:
test_eligible = eligible_subset(test)
if len(test_eligible) == 0:
    raise ValueError("No holdout rows have LAG_24. Need more route history for this comparison.")

predictions = pd.DataFrame({
    "DATE": test_eligible.DATE.to_numpy(),
    "ROUTE": test_eligible.ROUTE.to_numpy(),
    "OD_PAX": test_eligible.OD_PAX.to_numpy(),
    "Seasonal naive (24m)": test_eligible.LAG_24.to_numpy(),
})
final_models = {}
for name, estimator in models.items():
    pipeline = Pipeline([("prep", clone(transform)), ("model", clone(estimator))])
    pipeline.fit(train_val[features], train_val.OD_PAX)
    predictions[name] = np.clip(pipeline.predict(test_eligible[features]), 0, None)
    final_models[name] = pipeline

test_scores = []
for model_name in ["Seasonal naive (24m)", *models]:
    test_scores.append({"model": model_name, **score(predictions.OD_PAX, predictions[model_name])})
print(pd.DataFrame(test_scores).sort_values("MAE").to_string(index=False))
print("Preselected model:", winner)
print("Test coverage:", len(test_eligible), "of", len(test), "holdout route-months; missing-lag rows excluded from ALL comparable scores")

## 8 — Visual inspection and route-level error analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
monthly = predictions.groupby("DATE")[["OD_PAX", "Seasonal naive (24m)", winner]].sum()
ax.plot(monthly.index, monthly["OD_PAX"], label="Actual", marker="o")
ax.plot(monthly.index, monthly["Seasonal naive (24m)"], label="Baseline", marker=".")
if winner != "Seasonal naive (24m)":
    ax.plot(monthly.index, monthly[winner], label=winner, marker=".")
ax.set(title="Holdout demand (only eligible observed route-months)", ylabel="OD_PAX", xlabel="Target month")
ax.legend(); ax.grid(True, alpha=0.25); plt.xticks(rotation=35); plt.tight_layout(); plt.show()

rows = []
for route, part in predictions.groupby("ROUTE"):
    row = {"ROUTE": route, "n": len(part)}
    for name in ["Seasonal naive (24m)", *models]:
        row[name + "_MAE"] = score(part.OD_PAX, part[name])["MAE"]
    rows.append(row)
route_comparison = pd.DataFrame(rows).sort_values("ROUTE")
print(route_comparison.to_string(index=False))

## 9 — Save local results (never commit confidential data)

These files contain actual/predicted passenger demand; keep `data/processed/` Git-ignored. Check `git status` before committing notebooks.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
predictions.to_csv(OUTPUT_DIR / "aa_holdout_predictions_local.csv", index=False)
route_comparison.to_csv(OUTPUT_DIR / "aa_route_error_local.csv", index=False)
print("Saved local prediction and route-error CSVs (confidential).")
print("Do NOT commit data/processed/ or populated notebook outputs to GitHub.")

## 10 — Interpretation and next experiments

After running locally, write down (in NDA-compliant terms):

- Which model was selected using **validation MAE** (before holdout evaluation)?
- Did it beat the 24-month seasonal baseline on the holdout, and for which routes?
- How many route-months were excluded due to missing `LAG_24`?
- Does missing-month coverage limit what we can conclude?

**Next experiments (only with a fresh validation scheme):** market volatility, growth momentum, route-specific seasonality, geographic distance, and market-aware model selection. Add features only if computable at the issuance cutoff. With ~1,000 rows, prefer small, justified experiments over extensive tuning.

**Limitations:** The 24-month holdout evaluates *observed* route-months (not unobserved missing months). Aggregate plotted demand excludes ineligible rows. A full deployment forecast must create future route-month rows and calculate features from an explicit issuance-date snapshot. This notebook evaluates a historical 24-month scenario; it does not generate live future predictions.